# ViT-LENS: Notebook 01 — Setup & Dataset Preparation
**Research Goal**: Environment configuration, Google Drive mounting, dependency installation, and dataset pipeline initialization for ViT-LENS.


In [ ]:
# Step 1: Detect execution environment & mount Google Drive (if on Google Colab)
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    project_root = '/content/drive/MyDrive/ViT-LENS'
    os.makedirs(project_root, exist_ok=True)
    print(f'Mounted Google Drive at {project_root}')
except Exception:
    project_root = os.getcwd()
    print(f'Running locally / standard environment at {project_root}')

if project_root not in sys.path:
    sys.path.append(project_root)


In [ ]:
# Step 2: Install dependencies
!pip install -q timm torchvision pyyaml matplotlib pandas tqdm gradio


In [ ]:
# Step 3: Device detection & configuration verification
import torch
from src.utils import get_device, load_config, seed_everything

seed_everything(42)
device = get_device('auto')
print(f'Target Compute Device: {device}')
if device.type == 'cuda':
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('Running on CPU. For main SAE training, please switch Colab runtime to GPU (T4).')

config = load_config('config.yaml')
print('Configuration successfully loaded.')


### Dataset Loading & Class Mapping
ViT-LENS supports arbitrary image classification datasets with ImageFolder structure (`root/class_name/*.jpg`).
When using ImageNet-100 with an ImageNet-1K pretrained ViT, we correctly map the 100 subset classes to their true ImageNet-1K target indices.


In [ ]:
# Step 4: Prepare dataset (Generate prototype dataset for immediate execution or load custom)
from src.dataset import create_synthetic_prototype_dataset, build_dataloaders

data_dir = config['dataset']['data_dir'] + '/prototype'
print(f'Setting up dataset at: {data_dir}')

# Generate prototype samples if folder does not exist
create_synthetic_prototype_dataset(target_dir=data_dir, num_classes=5, images_per_class=40)

train_loader, val_loader, dataset = build_dataloaders(
    data_dir=data_dir,
    batch_size=config['dataset']['batch_size'],
    val_split=config['dataset']['val_split'],
    image_size=config['dataset']['image_size'],
    num_workers=0
)

print(f'Dataset size: {len(dataset)} images across {len(dataset.classes)} classes.')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
